# S2 Pass B · 1h rotate schedules (v2)

Infrastructure, not a hypothesis. Run after H-003 freezes `BAR_STAR = "1h"` (and after H-004), before H-005.
The daily Pass B in `s2_v2_pair_panel.ipynb` only writes `s2_rebalance_1d.csv`.

Writes:
- `s2_rebalance_1h.csv` — `quarterly` and `12m6m` selections, same schema as the 1d file.
- `s2_panel_1h_{train,full}.parquet` — rebuilt 1h union panel (`PAIRS_STAR` + freeze passers + every scheduled pair), so H-005 rotate arms have bars for the pairs they pick.

**Rotation pool:** the Pass A daily candidate screen filtered by `POOL_STAR` / `FDR_STAR` (same pool as the daily Pass B). That screen ends at `RESEARCH_IS_END` (2017), before any 1h bar, so it is causal for 1h rebalances.

**Units:** `lookback` is in sessions; the screen uses `L * BARS_PER_SESSION_1H` trailing 1h bars ending at the rebalance bar's close, and needs at least 252 sessions of bars (as the daily screen needs 252 days).
Yahoo serves about two years of 1h history, so `L = 504` sees all available bars (effectively the same window as `L = 252`).

**Timing:** rebalance at the close of the last 1h bar of the quarter / half-year; effective at the next 1h bar's open. Rebalances run only through the 1h research-IS end (70% of 1h sessions, same rule as `is_end_for_stack`).

Next: run H-005.

## 0. Imports & Config

In [ ]:
import os
import sys

import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import load_star_stack, require_star
from backtest.s2_coint.research import (
    BARS_PER_SESSION_1H,
    apply_bh_fdr,
    candidates_path,
    is_end_for_stack,
    load_s2_cuts,
    lookbacks_for_bar,
    overlap_is_end,
    panel_paths,
    rebalance_path,
    s2_artifacts_dir,
)
from backtest.s2_coint.rotation import (
    build_v2_rebalance_schedule,
    quarter_end_rebalance_dates,
    semiannual_rebalance_dates,
)
from data.ingestion.equity_fetcher import fetch_ohlcv
from data.processing.s2_coint_store import build_pair_panel

VERSION = "v2"
BAR = "1h"
STAR_PATH = os.path.join(s2_artifacts_dir(version=VERSION, root=ROOT), "s2_star_stack.json")
LOOKBACKS = (252, 504)  # sessions; same pre-registered L as the daily Pass B
COINT_PVALUE_THRESHOLD = 0.05
TOP_K = 50
PANEL_UNION_BUFFER = 80
SCREEN_OLS_WINDOW = lookbacks_for_bar(BAR)["ols_window"]  # 252 sessions of 1h bars
MIN_IS_BARS = 252 * BARS_PER_SESSION_1H
# 1h panel windows match H-003 so the rebuilt panel has the same columns / defaults.
PANEL_OLS_WINDOW = 63 * BARS_PER_SESSION_1H
PANEL_Z_WINDOW = 60 * BARS_PER_SESSION_1H
PANEL_HL_WINDOW = 252 * BARS_PER_SESSION_1H

cuts = load_s2_cuts(root=ROOT, version=VERSION)
stack = load_star_stack(STAR_PATH)
require_star("POOL_STAR", stack.get("POOL_STAR"))
require_star("FDR_STAR", stack.get("FDR_STAR"))
require_star("BAR_STAR", stack.get("BAR_STAR"))
if str(stack["BAR_STAR"]) != BAR:
    raise ValueError(f"BAR_STAR is {stack['BAR_STAR']!r}; use the daily Pass B in s2_v2_pair_panel.ipynb")
POOL_STAR = str(stack["POOL_STAR"])
FDR_STAR = str(stack["FDR_STAR"])
PAIRS_STAR = list(stack.get("PAIRS_STAR") or [])
if not PAIRS_STAR:
    raise ValueError("PAIRS_STAR empty — freeze H-001/H-002 first")
print("POOL_STAR", POOL_STAR, "FDR_STAR", FDR_STAR, "PAIRS_STAR", len(PAIRS_STAR))

## 1. Rotation pool (Pass A candidates)

In [ ]:
cand = pd.read_csv(candidates_path("1d", root=ROOT, version=VERSION))
if POOL_STAR == "industry":
    base = cand.loc[cand["same_industry"].astype(bool)].copy()
else:
    base = cand.loc[cand["same_sector"].astype(bool)].copy()
if FDR_STAR == "bh_fdr":
    base = apply_bh_fdr(base, alpha=0.05)
    base = base.loc[base["fdr_pass"]].copy()
else:
    base = base.loc[base["eligible"] & (base["pvalue"] < COINT_PVALUE_THRESHOLD)].copy()
base = base.sort_values("pvalue", kind="mergesort")

cand_pairs = [(str(r.ticker_y), str(r.ticker_x)) for r in base.itertuples(index=False)]
pool_pairs = list({tuple(sorted(p)): p for p in cand_pairs}.values())
star_pairs = [(p.split("|")[0], p.split("|")[1]) for p in PAIRS_STAR]
tickers = sorted(
    {t for p in pool_pairs for t in p} | {t for p in star_pairs for t in p}
)
print(f"pool pairs={len(pool_pairs)}  tickers={len(tickers)}")

## 2. Fetch 1h OHLC

In [ ]:
ohlc_1h: dict[str, pd.DataFrame] = {}
missing_1h: list[str] = []
for i, t in enumerate(tickers):
    raw = fetch_ohlcv(t, cuts["PANEL_START"], cuts.get("PANEL_END"), interval="1h", auto_adjust=False)
    if raw is None or raw.empty:
        missing_1h.append(t)
        continue
    fr = raw.copy()
    fr["date"] = pd.to_datetime(fr["date"])
    fr = fr.drop_duplicates("date").set_index("date")[["open", "high", "low", "close"]].astype(float).sort_index()
    ohlc_1h[t] = fr
    if (i + 1) % 50 == 0:
        print(f"fetched {i + 1}/{len(tickers)}")
if not ohlc_1h:
    raise RuntimeError("no 1h bars fetched — check yfinance / the 730-day window")
closes_1h = {t: fr["close"] for t, fr in ohlc_1h.items()}
print(f"1h names={len(ohlc_1h)}  missing={len(missing_1h)} {missing_1h[:20]}")

## 3. 1h research-IS end + rebalance dates

In [ ]:
bars = pd.DatetimeIndex(sorted({d for fr in ohlc_1h.values() for d in fr.index}))
IS_END_1H = is_end_for_stack(stack, pd.DataFrame({"date": bars}))
# Include the whole IS-end session so the last partial quarter rebalances at its final bar.
is_end_bar = bars[bars.normalize() <= IS_END_1H.normalize()].max()
q_dates = quarter_end_rebalance_dates(bars, end=is_end_bar)
s_dates = semiannual_rebalance_dates(bars, end=is_end_bar)
print("1h bars", bars.min(), "->", bars.max(), "| IS end session", IS_END_1H.date())
print("quarterly rebalances", [str(d) for d in q_dates])
print("12m6m rebalances", [str(d) for d in s_dates])

## 4. Rotate schedules → `s2_rebalance_1h.csv`

In [ ]:
reb = build_v2_rebalance_schedule(
    closes_1h,
    pool_pairs,
    bars,
    {"quarterly": q_dates, "12m6m": s_dates},
    LOOKBACKS,
    pool=POOL_STAR,
    fdr=FDR_STAR,
    ols_window=SCREEN_OLS_WINDOW,
    bars_per_session=BARS_PER_SESSION_1H,
    min_is_bars=MIN_IS_BARS,
    pvalue_threshold=COINT_PVALUE_THRESHOLD,
    top_k=TOP_K,
)
if reb.empty:
    raise RuntimeError("no 1h rebalance selections — not enough 1h history before the IS end")
reb_path = rebalance_path(BAR, root=ROOT, version=VERSION)
reb.to_csv(reb_path, index=False)
print("wrote", reb_path, "rows", len(reb))
print(reb.groupby(["schedule", "lookback", "rebalance_date"]).size())

## 5. 1h union panel → `s2_panel_1h_{train,full}.parquet`

In [ ]:
union_slots: set[tuple[str, str]] = set(star_pairs)
union_slots.update(pool_pairs[:PANEL_UNION_BUFFER])
union_slots.update(zip(reb["ticker_y"].astype(str), reb["ticker_x"].astype(str)))
union_pairs = sorted(union_slots)
print("1h union pairs", len(union_pairs))

panel_1h = build_pair_panel(
    ohlc_1h,
    union_pairs,
    ols_window=PANEL_OLS_WINDOW,
    z_window=PANEL_Z_WINDOW,
    hl_window=PANEL_HL_WINDOW,
    include_adf_pvalue=True,
)
panel_1h["date"] = pd.to_datetime(panel_1h["date"])
cut_1h = overlap_is_end(panel_1h, frac=0.70)
train_1h = panel_1h.loc[panel_1h["date"] <= cut_1h].copy()
train_p, full_p = panel_paths(BAR, root=ROOT, version=VERSION)
train_1h.to_parquet(train_p, index=False)
panel_1h.to_parquet(full_p, index=False)
print("wrote", train_p, train_1h.shape)
print("wrote", full_p, panel_1h.shape)
print("Next: run H-005.")

## 6. Notes

- Next: run H-005.
- Re-running overwrites `s2_rebalance_1h.csv` and the 1h panels; the 1h window slides with Yahoo's 730-day limit, so later runs see different dates.
- Do not open the sealed holdout here.